[Reference](https://blog.stackademic.com/i-rewrote-200-lines-of-pandas-code-into-20-here-is-how-193dda3335dd$0)

# What Method Chaining Actually Is

In [1]:
import pandas as pd

df = pd.read_csv("transactions.csv")
df = df.dropna()
df = df.rename(columns={"amt": "amount", "usr": "user_id"})
df = df[df["amount"] > 0]
df = df.assign(tax=df["amount"] * 0.18)
df = df.groupby("user_id")["amount"].sum().reset_index()
df = df.sort_values("amount", ascending=False)

In [2]:
df = (
    pd.read_csv("transactions.csv")
    .dropna()
    .rename(columns={"amt": "amount", "usr": "user_id"})
    .query("amount > 0")
    .assign(tax=lambda x: x["amount"] * 0.18)
    .groupby("user_id")["amount"]
    .sum()
    .reset_index()
    .sort_values("amount", ascending=False)
)

# The Script I Inherited

In [3]:
raw_data = pd.read_csv("sales.csv")

cleaned = raw_data.dropna(subset=["sale_id", "amount", "customer_id"])
cleaned = cleaned.drop_duplicates(subset=["sale_id"])
renamed = cleaned.rename(columns={
    "sale_id": "id",
    "amt": "amount",
    "cust_id": "customer_id",
    "dt": "date"
})
renamed["date"] = pd.to_datetime(renamed["date"])
renamed["month"] = renamed["date"].dt.month
renamed["year"] = renamed["date"].dt.year
filtered = renamed[renamed["amount"] > 0]
filtered = filtered[filtered["year"] == 2025]
filtered["category"] = filtered["amount"].apply(
    lambda x: "high" if x > 1000 else "medium" if x > 100 else "low"
)
summary = filtered.groupby(["customer_id", "month"]).agg(
    total_amount=("amount", "sum"),
    transaction_count=("id", "count")
).reset_index()
final = summary.sort_values(["customer_id", "month"])
final = final[final["total_amount"] > 500]

# The Rewrite

In [4]:
summary = (
    pd.read_csv("sales.csv")
    .dropna(subset=["sale_id", "amount", "customer_id"])
    .drop_duplicates(subset=["sale_id"])
    .rename(columns={
        "sale_id": "id",
        "amt": "amount",
        "cust_id": "customer_id",
        "dt": "date"
    })
    .assign(
        date=lambda x: pd.to_datetime(x["date"]),
        month=lambda x: x["date"].dt.month,
        year=lambda x: x["date"].dt.year,
        category=lambda x: pd.cut(
            x["amount"],
            bins=[0, 100, 1000, float("inf")],
            labels=["low", "medium", "high"]
        )
    )
    .query("amount > 0 and year == 2025")
    .groupby(["customer_id", "month"])
    .agg(
        total_amount=("amount", "sum"),
        transaction_count=("id", "count")
    )
    .reset_index()
    .sort_values(["customer_id", "month"])
    .query("total_amount > 500")
)

# pipe

In [5]:
def remove_outliers(df, column, threshold=3):
    mean = df[column].mean()
    std = df[column].std()
    return df[
        (df[column] >= mean - threshold * std) &
        (df[column] <= mean + threshold * std)
    ]

summary = (
    pd.read_csv("sales.csv")
    .dropna()
    .pipe(remove_outliers, column="amount")
    .groupby("customer_id")["amount"]
    .sum()
    .reset_index()
)